<a href="https://colab.research.google.com/github/mjcolebank/Colebank_REU_2026/blob/main/Model5_BayesianCNN_10layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Uncomment this line if Pyro is not installed.
%pip install pyro-ppl torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 15.8 MB/s eta 0:00:00


In [12]:
from re import X
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks

ECG_data_1K = np.load('ECG_waveforms_1K.npy')
# ECG_summary_1K = np.load('ECG_metrics_1K.npy')
Echo_5K = pd.read_csv('EchoNext_EchoData_5K.csv')

labels = Echo_5K['shd_moderate_or_greater_flag']


#initial definitions
negative = labels==0
positive = labels==1
y_ts = np.array(labels[:1000])
X = ECG_data_1K[:1000]
X_ts = np.swapaxes(X, 1, 2)
print(X_ts.shape)
print(y_ts.shape)


(1000, 12, 2500)
(1000,)


In [7]:
import math
import os
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.nn import PyroModule, PyroSample
from pyro.optim import Adam

try:
    from torchvision import datasets, transforms
    TORCHVISION_AVAILABLE = True
except Exception as e:
    TORCHVISION_AVAILABLE = False
    print("torchvision could not be imported:", e)

In [10]:
def set_seed(seed=0):
    """Set random seeds for reproducible examples."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
pyro.set_rng_seed(42)
pyro.clear_param_store()

# Use a GPU if one is available. Otherwise use the CPU.
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
print("Using device:", device)

Using device: cpu


In [13]:
import numpy as np
import matplotlib.pyplot as plt

import sympy
import sympy.core          # force submodule onto the namespace
import sympy.core.symbol
import torch               # now import torch

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

import importlib
sympy.core = importlib.import_module("sympy.core")
sympy.core.symbol = importlib.import_module("sympy.core.symbol")
print(hasattr(sympy, 'core'))

# Optional imports for MNIST.
# If torchvision is not installed, the synthetic time-series example will still run.

try:
    import torchvision
    import torchvision.transforms as transforms
    TORCHVISION_AVAILABLE = True
except ImportError:
    TORCHVISION_AVAILABLE = False

np.random.seed(42)
torch.manual_seed(42) #42

# Use a GPU if available. Otherwise, use CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


True
Using device: cpu


In [14]:
from re import X
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks

ECG_data_1K = np.load('ECG_waveforms_1K.npy')
ECG_summary_1K = np.load('ECG_metrics_1K.npy')
Echo_5K = pd.read_csv('EchoNext_EchoData_5K.csv')

labels = Echo_5K['shd_moderate_or_greater_flag']


#initial definitions
negative = labels==0
positive = labels==1
y_ts = np.array(labels[:1000])
X = ECG_data_1K[:1000]
X_ts = np.swapaxes(X, 1, 2)
print(X_ts.shape)
print(y_ts.shape)


(1000, 12, 2500)
(1000,)


In [15]:
labels = Echo_5K['shd_moderate_or_greater_flag']

y_ts = np.array(labels[:1000])
X = ECG_data_1K[:1000]
X_ts = np.swapaxes(X, 1, 2)

print(np.bincount(y_ts))
print("Majority-class baseline:", max(np.bincount(y_ts)) / len(y_ts))

[477 523]
Majority-class baseline: 0.523


In [16]:
#train & test
X_ts_tensor = torch.tensor(X_ts, dtype=torch.float32)
y_ts_tensor = torch.tensor(y_ts, dtype=torch.long)

ts_dataset = TensorDataset(X_ts_tensor, y_ts_tensor)

train_size = int(0.8 * len(ts_dataset))
test_size = len(ts_dataset) - train_size

ts_train_dataset, ts_test_dataset = random_split(
    ts_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

ts_train_loader = DataLoader(ts_train_dataset, batch_size=32, shuffle=True) #change batch size
batch = next(iter(ts_train_loader))
print(batch[0].shape)

ts_test_loader = DataLoader(ts_test_dataset, batch_size=128, shuffle=False)

print("Time-series training samples:", len(ts_train_dataset))
print("Time-series test samples:", len(ts_test_dataset))


torch.Size([32, 12, 2500])
Time-series training samples: 800
Time-series test samples: 200


In [17]:
class BayesianSmallCNN(PyroModule):
    def __init__(self, prior_scale=0.10, pooled_len=1, hidden_channels=150):
        super().__init__()
        self.pooled_len = pooled_len
        self.hidden_channels = hidden_channels

        # This buffer follows the model to whatever device it is moved to.
        self.register_buffer("_device_anchor", torch.tensor(0.0))

        def make_prior(shape):
            def prior(_module):
                device = self._device_anchor.device
                loc = torch.zeros((), device=device)
                scale = torch.tensor(prior_scale, device=device)
                return dist.Normal(loc, scale).expand(shape).to_event(len(shape))
            return prior

        # Ten convolution layers, defined with explicit names for Pyro stability.
        conv_specs = [
            (12, hidden_channels, 9, 2),
            (hidden_channels, hidden_channels, 3, 1),
            (hidden_channels, hidden_channels, 3, 1),
            (hidden_channels, hidden_channels, 3, 1),
            (hidden_channels, hidden_channels, 3, 1),
            (hidden_channels, hidden_channels, 3, 1),
            (hidden_channels, hidden_channels, 3, 1),
            (hidden_channels, hidden_channels, 3, 1),
            (hidden_channels, hidden_channels, 3, 1),
            (hidden_channels, hidden_channels, 3, 1),
        ]

        self.pool_every = 2
        self.num_convs = len(conv_specs)

        for i, (in_ch, out_ch, kernel_size, padding) in enumerate(conv_specs, start=1):
            conv = PyroModule[nn.Conv1d](
                in_ch,
                out_ch,
                kernel_size=kernel_size,
                padding=padding,
            )
            conv.weight = PyroSample(make_prior([out_ch, in_ch, kernel_size]))
            conv.bias = PyroSample(make_prior([out_ch]))
            setattr(self, f"conv{i}", conv)
            setattr(self, f"bn{i}", nn.BatchNorm1d(out_ch))

        # Adaptive pooling makes the classifier head independent of input length.
        self.fc1 = PyroModule[nn.Linear](hidden_channels, 32)
        self.fc1.weight = PyroSample(make_prior([32, hidden_channels]))
        self.fc1.bias = PyroSample(make_prior([32]))

        self.fc2 = PyroModule[nn.Linear](32, 2)
        self.fc2.weight = PyroSample(make_prior([2, 32]))
        self.fc2.bias = PyroSample(make_prior([2]))

    def forward(self, x, y=None, dataset_size=None):
        x = x.to(self._device_anchor.device)
        if y is not None:
            y = y.to(self._device_anchor.device)

        for i in range(1, self.num_convs + 1):
            conv = getattr(self, f"conv{i}")
            bn = getattr(self, f"bn{i}")

            x = F.relu(bn(conv(x)))

            # Pool every second layer, but not after the last conv.
            if i < self.num_convs and i % self.pool_every == 0:
                x = F.max_pool1d(x, kernel_size=2)

        x = F.adaptive_avg_pool1d(x, 1)
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)

        if y is not None and dataset_size is not None:
            scale = dataset_size / x.shape[0]
        else:
            scale = 1.0

        with pyro.plate("data", x.shape[0]):
            with pyro.poutine.scale(scale=scale):
                pyro.sample("obs", dist.Categorical(logits=logits), obs=y)

        return logits


In [18]:
pyro.clear_param_store()
set_seed(42)

bcnn = BayesianSmallCNN(prior_scale=2.0).to(device)
bcnn_guide = AutoDiagonalNormal(bcnn)

bcnn_optimizer = Adam({"lr": 1e-3,"betas": (0.9, 0.999)})
bcnn_svi = SVI(
    model=bcnn,
    guide=bcnn_guide,
    optim=bcnn_optimizer,
    loss=Trace_ELBO(),
)

In [19]:
def train_one_epoch_svi(svi, data_loader, device, dataset_size, max_batches=None):
    total_loss = 0.0
    total_examples = 0

    for batch_idx, (x, y) in enumerate(data_loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        x = x.to(device)
        y = y.to(device)

        loss = svi.step(x, y, dataset_size)
        total_loss += loss
        total_examples += x.shape[0]

    return total_loss / max(1, total_examples)

@torch.no_grad()
def predict_proba_bayesian(model, guide, x, num_samples=2500):
    """Average class probabilities over posterior samples of the network."""
    predictive = Predictive(
        model,
        guide=guide,
        num_samples=num_samples,
        return_sites=("_RETURN",),
    )
    out = predictive(x, None)

    # Shape is usually [num_samples, batch_size, num_classes].
    logits_samples = out["_RETURN"]
    probs_samples = torch.softmax(logits_samples, dim=-1)
    mean_probs = probs_samples.mean(dim=0)
    return mean_probs


@torch.no_grad()
def evaluate_bayesian_classifier(model, guide, data_loader, device, num_samples=2500, max_batches=None):
    """Evaluate posterior-averaged classification accuracy."""
    correct = 0
    total = 0

    for batch_idx, (x, y) in enumerate(data_loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        x = x.to(device)
        y = y.to(device)
        mean_probs = predict_proba_bayesian(model, guide, x, num_samples=num_samples)
        preds = mean_probs.argmax(dim=-1)
        correct += (preds == y).sum().item()
        total += y.numel()

    return correct / max(1, total)

In [20]:
num_epochs = 20
max_batches_per_epoch = None
num_prediction_samples = 15

bcnn_losses = []

for epoch in range(num_epochs):
    avg_loss = train_one_epoch_svi(
        bcnn_svi,
        ts_train_loader,
        device,
        dataset_size=train_size,
        max_batches=max_batches_per_epoch,
    )
    bcnn_losses.append(avg_loss)

    test_acc = evaluate_bayesian_classifier(
        bcnn,
        bcnn_guide,
        ts_test_loader,
        device,
        num_samples=num_prediction_samples,
        max_batches=5,
    )

    print(
        f"Epoch {epoch + 1:2d}/{num_epochs} | "
        f"avg negative ELBO per example = {avg_loss:.2f} | "
        f"approx test accuracy = {test_acc:.3f}"
    )

plt.figure(figsize=(7, 4))
plt.plot(bcnn_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Average negative ELBO per example")
plt.title("Bayesian CNN Training Loss")
plt.grid(True)
plt.show()


KeyboardInterrupt: 

In [ ]:
train_acc = evaluate_bayesian_classifier(
    bcnn,
    bcnn_guide,
    ts_train_loader,
    device,
    num_samples=100
)

test_acc = evaluate_bayesian_classifier(
    bcnn,
    bcnn_guide,
    ts_test_loader,
    device,
    num_samples=100
)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

Train accuracy: 0.78125
Test accuracy: 0.77


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import torch

@torch.no_grad()
def get_predictions(model, guide, data_loader, device, num_samples=250):
    all_preds = []
    all_labels = []

    for x, y in data_loader:
        x = x.to(device)
        y = y.to(device)

        mean_probs = predict_proba_bayesian(
            model, guide, x, num_samples=num_samples
        )
        preds = mean_probs.argmax(dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    return np.array(all_labels), np.array(all_preds)


y_true, y_pred = get_predictions(
    bcnn,
    bcnn_guide,
    ts_test_loader,
    device,
    num_samples=250
)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))